In [ ]:
# ============================================================
# RetailPulse | Phase 3: Data Cleaning
# Author: Naisha
# Date: May 2026
# Purpose: Fix data types, nulls, and quality issues
# ============================================================

import pandas as pd
import os

RAW="../data/raw/"
PROCESSED="../data/processed/"

tables = {
    "customers"   : "olist_customers_dataset.csv",
    "orders"      : "olist_orders_dataset.csv",
    "order_items" : "olist_order_items_dataset.csv",
    "payments"    : "olist_order_payments_dataset.csv",
    "products"    : "olist_products_dataset.csv",
    "sellers"     : "olist_sellers_dataset.csv",
    "reviews"     : "olist_order_reviews_dataset.csv",
    "geolocation" : "olist_geolocation_dataset.csv",
    "categories"  : "product_category_name_translation.csv"
}

dfs = {}
for name,files in tables.items():
    dfs[name]=pd.read_csv(os.path.join(RAW,files))
    print(f"✓ Loaded {name:15s} → {dfs[name].shape}")

In [ ]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    dfs['orders'][col] = pd.to_datetime(dfs['orders'][col])

print("✓ Date columns fixed")
print(dfs['orders'].dtypes)

## Cleaning Task 1 — Complete
- Fixed 5 date columns in orders table
- Converted from object (text) to datetime64
- Now able to do date math: delivery delays, monthly trends

In [ ]:
# Task 2: Fix products table
# 2a. Rename typo column
dfs['products'].rename(columns={
    'product_name_lenght': 'product_name_length', 
    'product_description_lenght': 'product_description_length'
}, inplace=True)

# 2b. Fill null category names with 'unknown'
dfs['products']['product_category_name'].fillna('unknown', inplace=True)

print("✓ Products table fixed")
print(f"Nulls remaining: {dfs['products'].isnull().sum().sum()}")
print(dfs['products'].columns.tolist())

## Cleaning Task 2 — Complete
- Renamed 2 typo columns in products table
- Filled 610 null category names with 'unknown'
- 1,838 nulls remaining in dimension columns (weight, size)

In [ ]:
dfs['payments'] = dfs['payments'][dfs['payments']['payment_type'] != 'not_defined']
print(len(dfs['payments']))

In [ ]:
dfs['payments'] = dfs['payments'][dfs['payments']['payment_installments'] > 0]
print(len(dfs['payments']))

## Cleaning Task 3 — Complete
- Dropped 3 rows where payment_type = 'not_defined'
- Dropped 2 rows where installments = 0
- Payments table now has 103,881 clean rows

In [ ]:
# Task 4a: Calculate delivery days
dfs['orders']['delivery_days'] = (
    dfs['orders']['order_delivered_customer_date'] - 
    dfs['orders']['order_purchase_timestamp']
).dt.days

print("✓ delivery_days column created")
print(dfs['orders']['delivery_days'].describe())

In [ ]:
# Task 4b: Calculate delivery delay
dfs['orders']['delivery_delay_days'] = (
    dfs['orders']['order_delivered_customer_date'] - 
    dfs['orders']['order_estimated_delivery_date']
).dt.days

print("✓ delivery_delay_days column created")
print(dfs['orders']['delivery_delay_days'].describe())

## Cleaning Task 4 — Complete
- Created delivery_days column (purchase → delivery)
- Created delivery_delay_days column (actual vs estimated)
- Average delivery = 12 days
- Average delay = -12 days (orders arrive early on average)
- Max delay = 188 days → severe outliers, watch in EDA

In [ ]:
# Task 5: Save cleaned tables
dfs['orders'].to_csv(PROCESSED + 'orders_clean.csv', index=False)
dfs['payments'].to_csv(PROCESSED + 'payments_clean.csv', index=False)
dfs['products'].to_csv(PROCESSED + 'products_clean.csv', index=False)

print("✓ Cleaned tables saved to data/processed/")